# Fine-Tuning the BERT Model

## Import Necessary Libraries and Define Custom Dataset Class

In [1]:
import sys
sys.path.append('C:/Users/thabi/OneDrive/Documents/FYP/advanced-crm-nlp-project/scripts')
import tokenization
import torch
from transformers import BertForSequenceClassification, Trainer, TrainingArguments
from torch.utils.data import Dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import numpy as np

Training Encodings - input_ids shape: torch.Size([25000, 512])
Testing Encodings - input_ids shape: torch.Size([25000, 512])
Sample Training Input IDs: tensor([  101, 22953,  2213,  4381,  2152,  9476,  4038,  2743,  2051,  3454,
         2082,  2166,  5089,  2086,  4252,  9518,  2599,  2903, 22953,  2213,
         4381, 26836, 18312,  2172,  3553,  4507,  5089, 25740,  5788, 13732,
        12369,  3993,  2493,  2156,  2157, 17203,  5089, 13433,  8737,  9004,
        10196,  4757,  2878,  3663, 10825,  2816,  2354,  2493,  2387,  2792,
         3076,  8385,  2699,  6402,  2082,  3202,  7383,  2152,  4438,  2240,
         7742, 10047, 12803,  2028,  5089,  3076,  6160, 22953,  2213,  4381,
         2152,  5987,  2116,  6001,  2287,  2228, 22953,  2213,  4381,  2152,
         2521, 18584,  2098, 12063,  3475,  2102,   102,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     

## Define a Custom Dataset Class

In [2]:
class IMDbDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        # Convert each encoding to a tensor
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        # Add labels
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

## Create Dataset Objects and Split Validation Set

In [3]:
from sklearn.model_selection import train_test_split

# Create Dataset objects
train_dataset = IMDbDataset(tokenization.train_encodings, tokenization.train_df['label'].tolist())
test_dataset = IMDbDataset(tokenization.test_encodings, tokenization.test_df['label'].tolist())

# Split training data into training and validation sets
train_indices, val_indices = train_test_split(
    list(range(len(train_dataset))),
    test_size=0.1,
    random_state=42
)

train_subset = torch.utils.data.Subset(train_dataset, train_indices)
val_subset = torch.utils.data.Subset(train_dataset, val_indices)

## Define Evaluation Metrics

In [4]:
def compute_metrics(pred):
    """
    Compute evaluation metrics.

    Parameters:
    - pred (transformers.EvalPrediction): Prediction results.

    Returns:
    - metrics (dict): Dictionary of computed metrics.
    """
    labels = pred.label_ids
    preds = np.argmax(pred.predictions, axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary')
    acc = accuracy_score(labels, preds)
    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

## Configure Training Arguments

In [5]:
import transformers
import accelerate
print(transformers.__version__)
print(accelerate.__version__)
%pip install transformers[torch] accelerate>=0.26.0

training_args = TrainingArguments(
    output_dir='./results',                  # Output directory
    num_train_epochs=3,                      # Number of training epochs
    per_device_train_batch_size=16,          # Batch size per device during training
    per_device_eval_batch_size=64,           # Batch size per device during evaluation
    warmup_steps=500,                        # Number of warmup steps for learning rate scheduler
    weight_decay=0.01,                       # Strength of weight decay
    logging_dir='./logs',                    # Directory for storing logs
    logging_steps=10,                        # Log every 10 steps
    evaluation_strategy='epoch',             # Evaluate at the end of each epoch
    save_strategy='epoch',                   # Save model at the end of each epoch
    load_best_model_at_end=True,             # Load the best model when finished training
    metric_for_best_model='accuracy',        # Use accuracy to evaluate the best model
    fp16=True                                 # Use mixed precision if available
)

4.46.3
1.1.1
Note: you may need to restart the kernel to use updated packages.


c:\Users\thabi\OneDrive\Documents\FYP\advanced-crm-nlp-project\venv\Lib\site-packages\transformers\training_args.py:1568: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


## Initialize the BERT Model for Sequence Classification

In [6]:
# Initialize the BERT model for binary classification
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


## Initialize the Trainer

In [7]:
trainer = Trainer(
    model=model,                             # The instantiated 🤗 Transformers model to be trained
    args=training_args,                      # Training arguments
    train_dataset=train_subset,              # Training dataset
    eval_dataset=val_subset,                 # Evaluation dataset
    compute_metrics=compute_metrics          # Function to compute metrics
)

## Start Training

In [ ]:
trainer.train()

  0%|          | 0/4221 [00:00<?, ?it/s]

C:\Users\thabi\AppData\Local\Temp\ipykernel_20720\1868611996.py:8: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}


{'loss': 0.718, 'grad_norm': 2.603797674179077, 'learning_rate': 1.0000000000000002e-06, 'epoch': 0.01}
{'loss': 0.748, 'grad_norm': 8.122150421142578, 'learning_rate': 2.0000000000000003e-06, 'epoch': 0.01}
{'loss': 0.7342, 'grad_norm': 3.7504162788391113, 'learning_rate': 3e-06, 'epoch': 0.02}
{'loss': 0.6965, 'grad_norm': 3.554870128631592, 'learning_rate': 4.000000000000001e-06, 'epoch': 0.03}
{'loss': 0.6891, 'grad_norm': 5.318952560424805, 'learning_rate': 5e-06, 'epoch': 0.04}
{'loss': 0.6792, 'grad_norm': 3.8335354328155518, 'learning_rate': 6e-06, 'epoch': 0.04}
{'loss': 0.6722, 'grad_norm': 6.572314262390137, 'learning_rate': 7.000000000000001e-06, 'epoch': 0.05}
{'loss': 0.6682, 'grad_norm': 4.0528974533081055, 'learning_rate': 8.000000000000001e-06, 'epoch': 0.06}
{'loss': 0.6178, 'grad_norm': 3.7492003440856934, 'learning_rate': 9e-06, 'epoch': 0.06}
{'loss': 0.643, 'grad_norm': 10.326457977294922, 'learning_rate': 1e-05, 'epoch': 0.07}
{'loss': 0.5587, 'grad_norm': 6.2783